In [10]:
!pip install langchain langchain-core langchain-community \
langchain-google-genai google-generativeai \
faiss-cpu pypdf sentence-transformers langdetect \
fastapi uvicorn python-telegram-bot nest-asyncio
!pip install -U langchain-text-splitters


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Import successful ✅")

ModuleNotFoundError: No module named 'langchain_text_splitters'

In [7]:
api_key="GOOGLE_API_KEY"
import os
import google.generativeai as genai
#load the secret key
api_key="API_KEY"
#configure the generative ai model
genai.configure(api_key=api_key)

ModuleNotFoundError: No module named 'google'

In [8]:
for model in genai.list_models():#models we can use,
  if "generateContent" in model.supported_generation_methods:#only text generation models
    print(model.name)

NameError: name 'genai' is not defined

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "Python Activity Pseudocode.pdf"

loader = PyPDFLoader(file_path)
documents = loader.load()

print("Pages loaded:", len(documents))

In [9]:
#Split the PDF into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Chunks created:", len(chunks))

ModuleNotFoundError: No module named 'langchain_text_splitters'

In [ ]:
#Create Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model loaded ✅")

In [ ]:
#import nece. libraries for doc. load and text splitting
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
 file_path= "Python Activity Pseudocode.pdf"

In [ ]:
loader=PyPDFLoader(file_path)#loader is like a reciever, will give pdf to documents

In [ ]:
documents= loader.load()

In [ ]:
print(f"loaded {len(documents)} pages from the pdf")

In [ ]:
print(documents) #holds with metadata

In [ ]:
print("the content of the first page:")
print(documents[0].page_content[:500])#500 tokens

In [ ]:
#create an instance of the text_splitter

In [ ]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50 ) #genereal ratio 10:1 100->10
#split the loader docs into the final chunks
chunks=text_splitter.split_documents(documents)
print(f"split into {len(chunks)} chunks")

In [ ]:
print("some content from the first chunk")
print(chunks[0].page_content)

In [ ]:
#own data , relevant chunks (divide)

In [ ]:
#importing the necessary libraries for embedding and vector store
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
model_name="sentence-transformers/all-MiniLM-L6-v2"
embeddings=HuggingFaceEmbeddings(model_name=model_name)
print("Local embedding model loaded !")

#Creating a Vector Store- The Brain!

In [ ]:
vectorstore= FAISS.from_documents(chunks,embeddings)
vectorstore.save_local("faiss_index")
print("The Vector Store is built and saved to the 'FAISS index' ")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI #llm
from langchain.prompts import PromptTemplate #prompt template
from langchain.chains import create_retrieval_chain #chain for retrieval
from langchain.chains.combine_documents import create_stuff_documents_chain #

In [ ]:
#load faiss vector store
vectorstore=FAISS.load_local("faiss_index",embeddings, allow_dangerous_deserialization= True)

In [ ]:
#initializing the LLM
llm=ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash", # Corrected model name
    temperature=0.7,
    google_api_key=api_key
 )#  (0-1)less , model more precise answer.High, model got more creative independence, 0.7=ideal
print("Component Loaded")

In [ ]:
#GPT
response = llm.invoke("What is Python?")
print(response.content)

In [ ]:
chat_history = [] 

In [ ]:
chat_history = []

while True:
    question = input("\nAsk a question (or type 'exit'): ")

    if question.lower() == "exit":
        break

    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    history_text = "\n".join(
        [f"User: {q}\nBot: {a}" for q, a in chat_history]
    )

    prompt = f"""
    Previous Conversation:
    {history_text}

    Context:
    {context}

    Question:
    {question}

    Answer only using the provided context.
    """

    response = llm.invoke(prompt)

    print("\nBot:", response.content)

    chat_history.append((question, response.content))

In [ ]:
def ask_rag(question):
    docs = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
    Answer using only the provided context.

    Context:
    {context}

    Question:
    {question}
    """

    response = llm.invoke(prompt)

    return response.content

In [ ]:
print(
    ask_rag(
        "How do I register a student?"
    )
)

In [ ]:
answer = ask_rag("What happens if a roll number already exists?")

print(answer)

In [ ]:
while True:
    question = input("\nAsk a question (or type 'exit'): ")

    if question.lower() == "exit":
        break

    print("\nAnswer:")
    print(ask_rag(question))

In [ ]:
#Defining the prompt template
prompt_template="""
Answer the question as thoroughly as possible based on the provided context.
If you dont know the answer then act a little cheezy and give a funny answer by teasing the person.
Make a conversation engaging and relatable by observing the person.

context:
{context}

Question: {input}

Answer:
"""
prompt=PromptTemplate(template=prompt_template,input_variables=["context","input"])

In [ ]:
#operating on LLM and Prompt
#create the document chain
document_chain=create_stuff_documents_chain(llm,prompt)
#create the retriever
retriever=vectorstore.as_retriever()
#retriever chain
retriever_chain=create_retrieval_chain(retriever, document_chain)

In [ ]:
question="Tell me about Space Science"
response= retriever_chain.invoke({"input":question})

print("Question:")
print(question)

print("\n--Answer--")
print(response["answer"])

In [ ]:
#import some necessary tools for multilingual conversations
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2" #model for multi lingual
embeddings_multi=HuggingFaceEmbeddings(model_name=model_name) #object
vectorstore_multi= FAISS.from_documents(chunks, embeddings_multi)
vectorstore_multi.save_local("faiss_index_multilingual")#saving
print("new vector store is created")

In [ ]:
#for the final pipeline for our multilingual model
from langdetect import detect
print("Language detection model tool imported")

In [ ]:
#1load the multilingual vector store
vectorstore_multi=FAISS.load_local("faiss_index_multilingual",embeddings_multi, allow_dangerous_deserialization= True)

In [ ]:
#2- create a new prompt template that include a language instruction
prompt_template_multi="""
Answer the question as thoroughly as possible based on the provided context.
If you don't know the answer , just say I don't have time for your extra bullshit, don't try to make up an answer.
Your final answer must be in the following langusage:(language)

Context:
{context}
Question: {input}

Answer:
"""

prompt_multi=PromptTemplate(
    template=prompt_template_multi,
    input_variables=["context","input","language"]


)

In [ ]:
# 3- Recreate the retrieval chain for the new components
retrieval_multi=vectorstore_multi.as_retriever()
document_chain_multi=create_stuff_documents_chain(llm, prompt_multi)
retrieval_chain_multi=create_retrieval_chain(retrieval_multi, document_chain)

In [ ]:
question_spanish="Cuéntame sobre el código de Python"
detect_language=detect(question_spanish)

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

docs = retriever.invoke("What is Python?")

print("Retrieved docs:", len(docs))
print(docs[0].page_content[:200])

In [ ]:
for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(model.name)

In [ ]:
response = llm.invoke("What is Python?")

print(response.content)

In [ ]:
detect_language

In [ ]:
response= retrieval_chain_multi.invoke({
    "input" : question_spanish,
    "language": detect_language
})
print(question_spanish)
print("\n------Answer------")
print(response["answer"])

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain


In [ ]:
# memory object which conatin the memory
memory= ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")

# new powerfull conversation chain
conversational_chian=ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retrieval_multi,
    memory=memory,
    return_source_documents=True #this is the key for getting sources back
)

#API

In [ ]:
# MORE FUNCTIONALITY
#convo based on historical data/context


In [ ]:
#We are adding pyngrok to our installation {cannot run on local host on colab environ.we need external environment/server  }
!pip install -q pyngrok

In [ ]:
# Import all necc libraries for this session
import nest_asyncio #nest_asyncio → Fixes a bug in Colab so we can run web servers inside it.
from fastapi import FastAPI #FastAPI → Makes a web app for your chatbot (like creating “doors” called endpoints).
from pydantic import BaseModel, Field #Pydantic (BaseModel) → Decides what kind of data can come in (example: only text questions, not random junk).
import uvicorn #uvicorn → The engine that runs your FastAPI app (so it’s live on the web).
import asyncio #asyncio → Helps handle many requests at the same time without blocking.
from pyngrok import ngrok #pyngrok → Creates a public link so others can use your chatbot (since Colab only runs locally).
from google.colab import userdata #google.colab.userdata → Safe place to keep your secret API keys in Colab.
import threading #threading → Runs the web server in the background while you keep using your notebook.

print("All necessary libraries for API and tunneling are imported")

#refer too notebok

In [ ]:
# defining our data models
# updating the base models from new conversational chain
class Source(BaseModel):
  content: str= Field(description="the actual text snippet from the source document")
  page: int| None= Field(description="the page number of the source doc., if available") #we will use field for better documentation
class QueryRequest(BaseModel):
  question: str
  language: str | None='en'
class QueryResponse(BaseModel):
  answer: str =Field(description="the generated answer from the chatbot")
  sources: list[Source]


#creating a fast api instance and apply the patch
app=FastAPI()
nest_asyncio.apply()#for multiple loops

print("app instance created")


In [ ]:

#end point that will recieve the question
@app.post("/query", response_model=QueryResponse)
async def process_query(request: QueryRequest):
  print(f"Recieved query: '{request.questio}' in language: '{request.language}' ")

  response= conversational_chain.invoke({
      "input": request.question,
      "language": request.language
  })

  #format the source docs. for the response
  formatted_sources=[]
  if response.get("source_documents"):
    for doc in response["source_documents"]:
      source= Source(content=doc.page_content, page=doc.metadata.get("page"))
      formatted_sources.append(source)

  return QueryResponse(answer=response["answer"], sources= formatted_source)



  #A simple root endpoint to check if server is running

  @app.post("/clear_memory")
  async def clear_memory():
    memory.clear()
    return {"message":"Memory has been cleared"}

  @app.get("/")
  async def read_root():
    return {"message": "Campus API is running"}

    print("api endpoints are defined")


more functionality ?

In [ ]:
# lets run the server and create a public url with ngrok cuz we are using collab

# set up the ngrok auth token
NGROK_AUTHTOKEN= userdata.get("NGROK_AUTH")
ngrok.set_auth_token(NGROK_AUTHTOKEN)

#define a func to  run a uvicorn server

def run_server():
  uvicorn.run(app, host="0.0.0.0", port= 8000)
# running the server in the saperate thread

server_thread= threading.thread(target=run_server)
server_thread.start()

# having buffer time to let the server start

import time
time.sleep(5)

# create a public url using ngrok
public_url= ngrok.connect(8000)

In [ ]:
#working on its goldfish memory today
# 1- Coversation Buffer Memory (tool from langchain)
# 2- Reliability(will tell the source , from where it has searced or extracted the answer)
#for it , chage retrieval chain to conversational retrieval chain
#change base model.
# change endpoints


In [ ]:
#(for the prev.notebook, refer to w  )

# How to test the full exp

In [ ]:
#How to test the full experience
# VS ------
#